In [1]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import numpy as np
import os
from io import BytesIO

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "processeddata"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

In [2]:
def load_blob_csv(blob_name):
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name
    )

    data = blob_client.download_blob().readall()
    return pd.read_csv(BytesIO(data))

# EKSEMPEL (tilpass til dine faktiske filer)
df_consumption = load_blob_csv("TIMENES_processed.csv")
df_weather = load_blob_csv("TIMENES_weather.csv")
df_norgespris = load_blob_csv("timenes_ts_norgespris.csv")

In [3]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../src")))

from analysis.regression import (
    prepare_norgespris_regression_data,
    fit_best_norgespris_model
)

regression_df = prepare_norgespris_regression_data(
    df_consumption,
    df_weather,
    df_norgespris,
    station_customers_total=None,
    exclude_consumption_codes=(26,),
)

results, analysis_df, metrics = fit_best_norgespris_model(regression_df)

print(f"Observasjoner totalt: {analysis_df.shape[0]:,}")
print(f"Brukt i modell: {metrics['nobs']:,}")
print(f"R2 (within): {metrics['r2_within']:.4f}")
print("Merk: modellen bruker log1p(value_kwh) = log(1 + value_kwh), så effekten gjelder 1 + forbruk.")
print(f"Gjennomsnittlig andel med Norgespris i etterperioden: {metrics['mean_share_post']:.2%}")
print(f"Effekt ved +10 prosentpoeng høyere andel: {metrics['effect_10pp_pct']:.2f}%")
print(f"Effekt ved 0% -> 100% andel: {metrics['effect_full_share_pct']:.2f}%")
print()
print(f"Total observert forbruk i etterperioden: {metrics['total_observed_post_kwh']:,.0f} kWh")
print(f"Estimert total effekt av Norgespris i etterperioden: {metrics['total_attributable_post_kwh']:,.0f} kWh")
print(f"Dette tilsvarer: {metrics['attributable_share_of_post_kwh_pct']:.2f}% av totalforbruket i etterperioden")
print(f"Per kunde i etterperioden: {metrics['attributable_per_customer_post_kwh']:.1f} kWh")
print(f"Per kunde per dag (etterperiode): {metrics['attributable_per_customer_post_day_kwh']:.3f} kWh")
print()
print(results.summary)
print()
print(f"Koeffisient for norgespris_share (log-skala): {metrics['beta_share']:.4f}")
print(f"Standardfeil: {results.std_errors.get('norgespris_share'):.4f}")
print(f"p-verdi: {results.pvalues.get('norgespris_share'):.4g}")

Observasjoner totalt: 14,011,002
Brukt i modell: 14,011,002
R2 (within): 0.1402
Merk: modellen bruker log1p(value_kwh) = log(1 + value_kwh), så effekten gjelder 1 + forbruk.
Gjennomsnittlig andel med Norgespris i etterperioden: 69.52%
Effekt ved +10 prosentpoeng høyere andel: 0.39%
Effekt ved 0% -> 100% andel: 3.96%

Total observert forbruk i etterperioden: 13,931,837 kWh
Estimert total effekt av Norgespris i etterperioden: 547,247 kWh
Dette tilsvarer: 3.93% av totalforbruket i etterperioden
Per kunde i etterperioden: 156.3 kWh
Per kunde per dag (etterperiode): 2.004 kWh

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_value_kwh   R-squared:                        0.1402
Estimator:                   PanelOLS   R-squared (Between):              0.0000
No. Observations:            14011002   R-squared (Within):               0.1402
Date:                Thu, Apr 23 2026   R-squared (Overall):              0.0501
Time:           

In [ ]:
import numpy as np

np.set_printoptions(suppress=False)
print(results.pvalues)

In [4]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../src")))

from analysis.weather_regression import fit_weather_before_after_model

pre_results, post_results, weather_pvalues_df = fit_weather_before_after_model(regression_df)

print("Værkoeffisienter før og etter Norgespris:")
print(weather_pvalues_df.to_string(index=False))

Værkoeffisienter før og etter Norgespris:
        variabel  beta_før  p_verdi_før  beta_etter  p_verdi_etter
 air_temperature -0.016093          0.0   -0.018753   0.000000e+00
precipitation_mm  0.009971          0.0    0.001263   6.537733e-07
      wind_speed  0.007853          0.0    0.010886   0.000000e+00
